# Gradient Boosting in Practice

**Companion lesson:** https://ml-viz.vercel.app/courses/ensemble-methods/03-xgboost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Gradient boosting from scratch

Depth-2 regression trees fit to residuals, scaled by a learning rate — the whole idea in 30 lines.

In [ ]:
def fit_stump(x, r, depth=2):
    # tiny recursive regression tree on 1D input
    if depth == 0 or len(x) < 8:
        return ('leaf', r.mean())
    best = None
    for s in np.quantile(x, np.linspace(0.1, 0.9, 17)):
        l, rgt = r[x <= s], r[x > s]
        if len(l) < 4 or len(rgt) < 4: continue
        sse = ((l - l.mean())**2).sum() + ((rgt - rgt.mean())**2).sum()
        if best is None or sse < best[0]: best = (sse, s)
    if best is None: return ('leaf', r.mean())
    s = best[1]
    return ('split', s, fit_stump(x[x <= s], r[x <= s], depth-1),
                       fit_stump(x[x > s],  r[x > s],  depth-1))

def predict_tree(t, x):
    if t[0] == 'leaf': return np.full_like(x, t[1], dtype=float)
    _, s, l, rgt = t
    out = np.empty_like(x, dtype=float)
    out[x <= s] = predict_tree(l, x[x <= s]); out[x > s] = predict_tree(rgt, x[x > s])
    return out

In [ ]:
x = np.sort(np.random.rand(120) * 6)
y = np.sin(x) + 0.25 * np.random.randn(120)
x_test = np.linspace(0, 6, 300); y_test_true = np.sin(x_test)

def boost(eta, n_trees):
    pred, pred_test, trees = np.zeros_like(y), np.zeros_like(x_test), []
    test_err = []
    for _ in range(n_trees):
        t = fit_stump(x, y - pred)            # fit residuals = negative gradient of MSE
        pred += eta * predict_tree(t, x)
        pred_test += eta * predict_tree(t, x_test)
        test_err.append(np.mean((pred_test - y_test_true) ** 2))
    return pred_test, test_err

## Shrinkage in action: η = 1.0 vs η = 0.05

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
p_fast, err_fast = boost(eta=1.0, n_trees=200)
p_slow, err_slow = boost(eta=0.05, n_trees=200)

axes[0].scatter(x, y, s=12, c='#475569')
axes[0].plot(x_test, p_fast, color='#f43f5e', lw=2, label='η=1.0 (200 trees)')
axes[0].plot(x_test, p_slow, color='#14b8a6', lw=2, label='η=0.05 (200 trees)')
axes[0].plot(x_test, y_test_true, color='white', lw=1, ls=':'); axes[0].legend()

axes[1].plot(err_fast, color='#f43f5e', label='η=1.0')
axes[1].plot(err_slow, color='#14b8a6', label='η=0.05')
axes[1].set_xlabel('trees'); axes[1].set_ylabel('test MSE'); axes[1].legend()
plt.tight_layout(); plt.show()
# η=1.0 overfits quickly: test error bottoms out early then climbs
# η=0.05 descends slowly to a lower minimum — this is why small η wins

**Try it:** implement early stopping — track the best test MSE and stop after 25 trees without improvement. Then try the real thing: `pip install xgboost` and compare `eta=0.3` vs `eta=0.03` with `early_stopping_rounds=50` on any tabular dataset.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Early stopping

The "Try it" above asked for exactly this. Given the test-error curve across boosting rounds, find the best round, and implement patience-based early stopping: stop at the first round that is `patience` rounds past the best so far.

In [ ]:
def best_iteration(test_mse):
    """Index of the round with the lowest test error."""
    # TODO(you): argmin
    return ...


def early_stopping_round(test_mse, patience):
    """First round that is `patience` rounds past the best so far
    (or the last round if never triggered)."""
    best = np.inf
    best_i = 0
    for i, v in enumerate(test_mse):
        # TODO(you): if v improves on best, record (best, best_i);
        # otherwise stop once i - best_i >= patience
        ...
    return len(test_mse) - 1

In [ ]:
# Checks — run me
curve = [5.0, 4.0, 3.0, 2.5, 2.6, 2.7, 2.8]
assert best_iteration(curve) == 3, "test error bottoms out at round 3"
assert early_stopping_round(curve, patience=2) == 5, "no improvement for 2 rounds after 3 -> stop at 5"
assert early_stopping_round([5.0, 4.0, 3.0], patience=5) == 2, "never triggered -> run to the end"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def best_iteration(test_mse):
    return int(np.argmin(test_mse))


def early_stopping_round(test_mse, patience):
    best = np.inf
    best_i = 0
    for i, v in enumerate(test_mse):
        if v < best:
            best, best_i = v, i
        elif i - best_i >= patience:
            return i
    return len(test_mse) - 1
```

</details>

### Exercise 2 — XGBoost's leaf weight

XGBoost picks each leaf's value in closed form from the sums of first and second derivatives of the loss over the points in that leaf:

$$w^* = -\frac{G}{H + \lambda}, \qquad \text{gain} = \frac{1}{2} \, \frac{G^2}{H + \lambda}$$

Implement both. With $\lambda = 0$ this is a pure Newton step; the checks show how $\lambda$ shrinks leaves toward zero **and** lowers split gains — regularization built into the tree itself.

In [ ]:
def leaf_weight(G, H, lam):
    """Optimal leaf value from gradient sum G, Hessian sum H, regularization lam."""
    # TODO(you): -G / (H + lam)
    return ...


def leaf_gain(G, H, lam):
    """Quality score of the leaf: 0.5 * G^2 / (H + lam)."""
    # TODO(you)
    return ...

In [ ]:
# Checks — run me
assert abs(leaf_weight(-10.0, 5.0, 1.0) - 10 / 6) < 1e-12, "w* = -G/(H + lambda)"
assert abs(leaf_weight(-10.0, 5.0, 0.0) - 2.0) < 1e-12, "lambda = 0: the raw Newton step"
assert abs(leaf_weight(-10.0, 5.0, 1e9)) < 1e-7, "huge lambda crushes the leaf toward 0"
assert leaf_gain(-10.0, 5.0, 1.0) < leaf_gain(-10.0, 5.0, 0.0), "regularization also lowers the split gain"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def leaf_weight(G, H, lam):
    return -G / (H + lam)


def leaf_gain(G, H, lam):
    return 0.5 * G ** 2 / (H + lam)
```

</details>